# 5. Análise Estatística - Covariância, Correlações e Testes de Hipóteses

Este notebook aborda as secções 3.5 e 3.6 do relatório, focando-se na variação conjunta entre variáveis quantitativas e na validação estatística das hipóteses definidas.

In [1]:
import pandas as pd
from scipy import stats
from IPython.display import display

# Carregamento da base de dados (primeiros 300.000 registos)
df = pd.read_csv('../data/instagram_usage_lifestyle.csv')
df = df.head(300000)

# Variáveis chave para a análise bivariada
analysis_cols = [
    'daily_active_minutes_instagram', 
    'perceived_stress_score', 
    'self_reported_happiness', 
    'sleep_hours_per_night', 
    'reels_watched_per_day', 
    'exercise_hours_per_week'
]

df_subset = df[analysis_cols].dropna()

## 5.1 Covariância

A matriz de covariância permite analisar a direção da variação conjunta entre as variáveis de uso do Instagram e de bem-estar.

In [2]:
cov_matrix = df_subset.cov()
print("Matriz de Covariância:")
display(cov_matrix.round(3))

Matriz de Covariância:


,daily_active_minutes_instagram,perceived_stress_score,self_reported_happiness,sleep_hours_per_night,reels_watched_per_day,exercise_hours_per_week
daily_active_minutes_instagram,12125.416,1086.837,-117.754,-0.120,7334.834,0.405
perceived_stress_score,1086.837,139.920,0.010,-0.006,608.041,-0.033
self_reported_happiness,-117.754,0.010,8.241,0.006,-64.265,-0.057
sleep_hours_per_night,-0.120,-0.006,0.006,1.209,-0.190,0.007
reels_watched_per_day,7334.834,608.041,-64.265,-0.190,5802.804,0.170
exercise_hours_per_week,0.405,-0.033,-0.057,0.007,0.170,15.925


## 5.2 Correlação de Pearson e Spearman

Cálculo dos coeficientes de correlação (Pearson para relações lineares, Spearman para relações monotónicas) para os pares de variáveis definidos nas secções 3.5.2 e 3.5.3.

In [3]:
correlation_pairs = [
    ('daily_active_minutes_instagram', 'perceived_stress_score'),
    ('daily_active_minutes_instagram', 'self_reported_happiness'),
    ('daily_active_minutes_instagram', 'sleep_hours_per_night'),
    ('reels_watched_per_day', 'perceived_stress_score'),
    ('exercise_hours_per_week', 'self_reported_happiness'),
    ('sleep_hours_per_night', 'self_reported_happiness')
]

results = []

for v1, v2 in correlation_pairs:
    pearson_r, p_pearson = stats.pearsonr(df_subset[v1], df_subset[v2])
    spearman_rho, p_spearman = stats.spearmanr(df_subset[v1], df_subset[v2])
    
    results.append({
        'Variável 1': v1,
        'Variável 2': v2,
        'Pearson (r)': round(pearson_r, 4),
        'P-value (Pearson)': f"{p_pearson:.4e}",
        'Spearman (rho)': round(spearman_rho, 4),
        'P-value (Spearman)': f"{p_spearman:.4e}"
    })

df_correlations = pd.DataFrame(results)
print("Resultados das Correlações:")
display(df_correlations)

Resultados das Correlações:


,Variável 1,Variável 2,Pearson (r),P-value (Pearson),Spearman (rho),P-value (Spearman)
0,daily_active_minutes_instagram,perceived_stress_score,0.8344,0.0000e+00,0.8406,0.0000e+00
1,daily_active_minutes_instagram,self_reported_happiness,-0.3725,0.0000e+00,-0.3596,0.0000e+00
2,daily_active_minutes_instagram,sleep_hours_per_night,-0.0010,5.8646e-01,-0.0012,5.1296e-01
3,reels_watched_per_day,perceived_stress_score,0.6748,0.0000e+00,0.6732,0.0000e+00
4,exercise_hours_per_week,self_reported_happiness,-0.0050,6.6546e-03,-0.0055,2.4956e-03
5,sleep_hours_per_night,self_reported_happiness,0.0020,2.6816e-01,0.0021,2.4048e-01


## 5.3 Testes de Hipóteses

Validação das três hipóteses formuladas na secção 3.6 do relatório.

### Hipótese 1: Stress e Uso do Instagram
* **H₀:** Não existe diferença significativa no tempo de uso entre utilizadores com alto e baixo stress.
* **H₁:** Utilizadores com maior stress usam o Instagram por mais tempo.

In [4]:
stress_median = df_subset['perceived_stress_score'].median()

high_stress = df_subset[df_subset['perceived_stress_score'] > stress_median]['daily_active_minutes_instagram']
low_stress = df_subset[df_subset['perceived_stress_score'] <= stress_median]['daily_active_minutes_instagram']

t_stat1, p_val1 = stats.ttest_ind(high_stress, low_stress, equal_var=False)

print(f"Média Tempo (Alto Stress): {high_stress.mean():.2f} min")
print(f"Média Tempo (Baixo Stress): {low_stress.mean():.2f} min")
print(f"Estatística t: {t_stat1:.3f} | P-value: {p_val1:.4e}")

Média Tempo (Alto Stress): 270.41 min
Média Tempo (Baixo Stress): 110.44 min
Estatística t: 577.726 | P-value: 0.0000e+00


### Hipótese 2: Exercício e Felicidade
* **H₀:** Não existe diferença significativa na felicidade entre utilizadores ativos e sedentários.
* **H₁:** Utilizadores mais ativos reportam maior felicidade.

In [5]:
exercise_median = df_subset['exercise_hours_per_week'].median()

active_group = df_subset[df_subset['exercise_hours_per_week'] > exercise_median]['self_reported_happiness']
sedentary_group = df_subset[df_subset['exercise_hours_per_week'] <= exercise_median]['self_reported_happiness']

t_stat2, p_val2 = stats.ttest_ind(active_group, sedentary_group, equal_var=False)

print(f"Média Felicidade (Ativos): {active_group.mean():.2f}")
print(f"Média Felicidade (Sedentários): {sedentary_group.mean():.2f}")
print(f"Estatística t: {t_stat2:.3f} | P-value: {p_val2:.4e}")

Média Felicidade (Ativos): 5.47
Média Felicidade (Sedentários): 5.51
Estatística t: -3.579 | P-value: 3.4460e-04


### Hipótese 3: Sono e Felicidade
* **H₀:** Não existe correlação significativa entre as horas de sono e a felicidade.
* **H₁:** Existe uma correlação positiva significativa.

In [6]:
rho, p_val3 = stats.spearmanr(df_subset['sleep_hours_per_night'], df_subset['self_reported_happiness'])

print(f"Correlação Spearman (rho): {rho:.4f}")
print(f"P-value: {p_val3:.4e}")

Correlação Spearman (rho): 0.0021
P-value: 2.4048e-01


# ANÁLISE CRÍTICA

## Interpretação dos Resultados

Os testes de hipóteses revelam associações significativas entre o uso do Instagram e indicadores de bem-estar, como stress e felicidade. No entanto, a natureza sintética do dataset limita a validade externa dos achados. Correlações observadas (e.g., positiva entre tempo no Instagram e stress) podem ser artificiais, resultantes dos algoritmos de geração de dados.

## Limitações

- **Causalidade vs. Correlação**: Os testes não estabelecem causalidade; fatores externos podem influenciar ambas as variáveis.

- **Dados Sintéticos**: Resultados não generalizam para populações reais, onde vieses de autorrelato e seleção são comuns.

- **Variáveis Não Incluídas**: Fatores como idade, gênero ou contexto socioeconômico não foram controlados adequadamente.

Recomenda-se estudos longitudinais com dados reais para confirmar essas associações e explorar mecanismos subjacentes ao impacto das redes sociais no bem-estar psicológico.